### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [33]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [34]:
### Read all PDF's from directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #Find all PDF files recursively

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #add source info to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata["source_path"] = str(pdf_file)
                doc.metadata['file_type'] = 'pdf'     

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all the documents in the PDF directory
all_pdf_documents = process_all_pdfs("../data")


Found 2 PDF files to process

Processing: Aman Agrahari 5SFS.pdf
 Loaded 1 pages

Processing: Aman Agrahari FS4S.pdf
 Loaded 1 pages

Total documents loaded: 2


In [35]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [36]:
chunks = split_documents(all_pdf_documents)
chunks

Split 2 documents into 9 chunks

Example chunk:
Content: Aman Agrahari
7234909407
G I T H U B | |
EDUCATION
EXPERIENCE
Web3Task  
Full Stack Developer Internship 
Sept 2024 - Aug 2028
April 2025 - Present
April 2026 - May 2026
aman8cse@gmail.com |
PRODUCTS ...
Metadata: {'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-06-03T14:15:36+00:00', 'title': 'resume template.pdf', 'moddate': '2026-06-03T14:15:36+00:00', 'keywords': 'DAHELWz6xSo,BAGZGbj1AHg,0', 'author': 'Aman', 'trapped': '/False', 'source': '..\\data\\pdf\\Aman Agrahari 5SFS.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Aman Agrahari 5SFS.pdf', 'source_path': '..\\data\\pdf\\Aman Agrahari 5SFS.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-06-03T14:15:36+00:00', 'title': 'resume template.pdf', 'moddate': '2026-06-03T14:15:36+00:00', 'keywords': 'DAHELWz6xSo,BAGZGbj1AHg,0', 'author': 'Aman', 'trapped': '/False', 'source': '..\\data\\pdf\\Aman Agrahari 5SFS.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Aman Agrahari 5SFS.pdf', 'source_path': '..\\data\\pdf\\Aman Agrahari 5SFS.pdf', 'file_type': 'pdf'}, page_content='Aman Agrahari\n7234909407\nG I T H U B | |\nEDUCATION\nEXPERIENCE\nWeb3Task  \nFull Stack Developer Internship \nSept 2024 - Aug 2028\nApril 2025 - Present\nApril 2026 - May 2026\naman8cse@gmail.com |\nPRODUCTS & PROJECTS\nWatchroom - Real-Time Watch Party Platform\nLIC Calc - Insurance Analytics & Advisory Engine\nEngineered a real-time watch party platform enabling synchronized YouTube playback, live chat, reactions.\nBuilt room ownership, moderator controls, participant permissions, and session lifecycl

In [37]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [38]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self,texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        Args:
            texts: List of text strings to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not found")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
#Initialize embedding manager

embedding_manager = EmbeddingManager();
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2609.61it/s]


Model loaded successfully. Embedding dimension: 384


### Vector Store

In [41]:
class VectorStore:
    """Manages document embeddings in a chromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        Args:
            collection_name: Name of the chromaDB collection
            persist_directory: Directory to persist vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize chromaDB client and collection"""
        try:
            #Create persistent chromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"document": "PDF document embeddingsfor RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initilizing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add document and their embeddings to vector store
        Args:
            documents: List of LangChain documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents should be equal to number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for chromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embeddings
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to collection in vector store {e}")
            raise

vectorstore = VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 72


In [ ]:
### Convert text to embeddings
texts = [doc.page_content for doc in chunks]

### Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

### Store in the Vector Store now
vectorstore.add_documents(chunks, embeddings)